# GitHub Sentiment Monitor — запуск в Google Colab

До выполнения ячеек откройте `Среда выполнения → Сменить среду выполнения` и при необходимости выберите GPU (например, T4 GPU). Загрузите ZIP-архив проекта, установите зависимости и запустите Streamlit через встроенный прокси Google Colab. В приложении можно перечислять репозитории через запятую или с новой строки.


In [ ]:
from google.colab import files
uploaded = files.upload()  # Выберите github-sentiment-monitor.zip
archive_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if archive_name is None:
    raise ValueError("Загрузите ZIP-архив проекта.")
print("Загружен архив:", archive_name)


In [ ]:
import zipfile
from pathlib import Path

with zipfile.ZipFile(archive_name) as archive:
    archive.extractall("/content")

# Ищем app.py, не привязываясь к имени корневой папки ZIP-архива.
candidates = [path.parent for path in Path("/content").glob("*/app.py")]
if not candidates:
    raise FileNotFoundError("В архиве не найдена папка проекта с app.py.")
project_dir = next((p for p in candidates if p.name in {"github-sentiment-monitor", "github_sentiment_streamlit"}), candidates[0])
%cd $project_dir
print("Папка проекта:", project_dir)


In [ ]:
!pip -q install jedi -r requirements.txt

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Устройство:", torch.cuda.get_device_name(0))
else:
    print("Будет использоваться CPU. Для GPU выберите GPU-ускоритель в настройках среды выполнения Colab и перезапустите ноутбук.")


## Проверка GPU и запуск Streamlit

После установки зависимостей следующая ячейка проверит, видит ли PyTorch CUDA GPU. Если GPU доступен, классификатор тональности автоматически использует его; иначе приложение останется на CPU. Затем ноутбук запустит Streamlit и откроет интерфейс через встроенный прокси Colab. LocalTunnel не используется. Не публикуйте GitHub token в ноутбуке или открытых ячейках — вводите его непосредственно в поле приложения.


In [ ]:
import os
import subprocess
import time
from pathlib import Path
import requests
from google.colab import output

# После перезапуска среды Colab рабочая директория может сброситься.
project_candidates = [path.parent for path in Path("/content").glob("*/app.py")]
if not project_candidates:
    raise FileNotFoundError(
        "Не найдена папка проекта с app.py. "
        "Повторно выполните ячейки загрузки и распаковки ZIP-архива."
    )
project_dir = next(
    (path for path in project_candidates if path.name in {"github-sentiment-monitor", "github_sentiment_streamlit"}),
    project_candidates[0],
)
os.chdir(project_dir)
print("Рабочая директория:", Path.cwd())

# Если Streamlit уже запущен (например, ячейку выполняют повторно), используем его.
health_url = "http://127.0.0.1:8501/_stcore/health"
ready = False
try:
    ready = requests.get(health_url, timeout=2).status_code == 200
except requests.RequestException:
    pass

if not ready:
    log_path = "/tmp/streamlit.log"
    log_file = open(log_path, "w")
    process = subprocess.Popen(
        [
            "streamlit", "run", "app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
        cwd=str(project_dir),
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

    for _ in range(30):
        if process.poll() is not None:
            break
        try:
            response = requests.get(health_url, timeout=2)
            if response.status_code == 200:
                ready = True
                break
        except requests.RequestException:
            pass
        time.sleep(2)

if ready:
    print("Streamlit успешно запущен. Открываю интерфейс через прокси Colab...")
    output.serve_kernel_port_as_window(8501)
else:
    print("Не удалось запустить Streamlit. Журнал запуска:")
    print(Path("/tmp/streamlit.log").read_text(errors="replace"))
